<a href="https://colab.research.google.com/github/napsugark/LLM_Course/blob/main/05_LLM_Learning_Path_6_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# **Monitoring and Evaluation**

Link to older version: https://colab.research.google.com/drive/1GrHVkwXcx7aT1Ixw5KLb7ZHm6dz4oETT

# Concepts to know


At the end of this module you should have an understanding of the following metrics commonly used in evaluation of LLM solutions:

- Classification Metrics
 - Precision
 - Accuracy
 - Recall
 - F1-Score
- Text similarity metrics
 - Levensthein
 - Semantic Similarity
- LLM-as-a-Judge
- RAG Metrics
 - Faithfullnes
 - Answer Relevancy
 - Context Precision
 - Context Recall


# Materials


### Mandatory

Introduction to LangSmith Course (3.5 hrs):
- https://academy.langchain.com/courses/intro-to-langsmith

Note: To follow along on this course you will need to have a LangSmith account You can make a free version with 5,000 traces per months. Instead of an OpenAI Key use an AzureOpenAI key (and replace the respective clients in the code).

LLM Metrics and Evaluation:
- https://www.confident-ai.com/blog/llm-evaluation-metrics-everything-you-need-for-llm-evaluation
- https://learn.microsoft.com/en-us/ai/playbook/technology-guidance/generative-ai/working-with-llms/evaluation/list-of-eval-metrics#metrics-for-rag-pattern
- https://cohere.com/blog/classification-eval-metrics






### Optional
- Paper: Can LLMs Be an ALternative to Human Evaluation: https://arxiv.org/pdf/2305.01937v1
- Build external evaluation pipelines with langfuse: https://langfuse.com/guides/cookbook/example_external_evaluation_pipelines
- https://symbl.ai/developers/blog/an-in-depth-guide-to-benchmarking-llms/-

- Benchmarks: https://www.confident-ai.com/blog/the-current-state-of-benchmarking-llms#still-problems-with-benchmarking-llms-remain

# Coding




## Langfuse:

In [ ]:
pip install langfuse openai chainlit

### Example:  Trace calls in langfuse



Create a free langfuse account and a project for this LLM Learning Path. Then integrate langfuse tracking into your Movie RAG chatbot by:
- pip install langfuse
- add the following environment variables to .env
- replace the the AzureOpenAI import statement with from langfuse.openai import AzureOpenAI

### Example: Add a session ID

In [ ]:
import uuid
import chainlit as cl
# Generate a session id for each chat session inside the reset_session() function by adding
cl.user_session.set("session_id", str(uuid.uuid4()))

# Include the session_id as parameter for each completion call by passing it as the parameter to the completions.create call
stream = client.chat.completions.create(
            model='gpt-4o-mini',
            messages=[
                {"role": "system", "content": get_system_prompt(cl.user_session.get("language", "English"))},
                {"role": "user", "content": f"QUESTION: {reformulated_question}"},
                {"role": "system", "content": f"CONTEXT: {context}"}],
            temperature=cl.user_session.get("temperature", 0),
            stream=True,
            session_id=cl.user_session.get("session_id"))


### Assiggnment: Add a user id

In [ ]:
# add user_id parameter to the completions.create() call

### Assignment: Provide a name to each of the calls

In [ ]:
# add name parameter to the completions.create() call

### Example: Add user feedback and track it in langfuse

Langfuse offers the option to add scores (such as a user feedback) to a trace. To implement this we need swith from the drop-in replacement to the @observe decorator, applied to a function calling the LLM. This function will return the response of the LLM alongside the trace id, so that we can use it to add user feedback to the trace in langfuse.

In [ ]:
@cl.on_message
@observe(capture_output=False)
async def message_send(message: cl.Message):
    trace_id = langfuse_context.get_current_trace_id()
    language = cl.user_session.get("language", "English")
    temperature = cl.user_session.get("temperature", 0)
    chat_history = cl.user_session.get("chat_history", [])
    retriever = cl.user_session.get("retriever")
    client = cl.user_session.get("client")
    retrieved_docs = retriever.similarity_search(message.content, k=4)
    context = format_docs(retrieved_docs)
    system_prompt = get_system_prompt(language)
    chat_history.append({"role": "system", "content": system_prompt})
    chat_history.append({"role": "user", "content": f"QUESTION: {message.content}"})
    chat_history.append({"role": "system", "content": f"CONTEXT: {context}"})
    full_response = ""
    source_elements = []
    stream = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=temperature,
        stream=True,
        messages=[
            {"role": m["role"], "content": m["content"]}
            for m in chat_history
        ]
    )

    msg = cl.Message(content="")
    await msg.send()
    for chunk in stream:
        if not chunk.choices or not chunk.choices[0].delta:
            continue

        delta = chunk.choices[0].delta.content or ""
        full_response += delta
        await msg.stream_token(delta)
    await msg.update()

    source_elements.append(cl.Text(content=context, name="Context", display="side"))
    msg.content += "\n\nContext "
    msg.elements = source_elements
    await msg.update()

    chat_history.append({"role": "assistant", "content": full_response})
    cl.user_session.set("chat_history", chat_history)
    langfuse_context.update_current_observation(
        output=full_response
    )

Include this code at the end of the script. It will give users the option to provide feedback on the last response of the LLM. Try it out and find your feedback in the langfuse UI.

In [ ]:
  if len(cl.user_session.get("chat_history")) > 2:
          feedback_msg = await cl.AskActionMessage(
              content="Was this response helpful?",
              actions=[
                  cl.Action(name="feedback", payload={"value": "👍"}, label="👍"),
                  cl.Action(name="feedback", payload= {"value": "👎"}, label="👎")
              ]
          ).send()

          value = feedback_msg.get("payload").get("value")
          if trace_id:
              langfuse_client = Langfuse()
              langfuse_client.score(
                  trace_id=trace_id,
                  name="user-feedback",
                  value=value
              )
          await cl.Message(content="Thank you for your feedback!").send()


### Assignment: Create an LLM Judge via for hallucination in langfuse UI

Following this [walkthrough](https://langfuse.com/docs/scores/model-based-evals) for the LLM-as-a Judge for Traces, set up a evaluator in the UI that evaluates hallucination for all incoming traces related to generating a chatbot answer (not the reformulation step).

After you are done, deactivate the evaluator.

### Assignment: Create an LLM Judge for context correctness via Python SDK